### Import packages

In [3]:
import pandas as pd
from scipy.stats import spearmanr
from scipy.stats import chi2_contingency
import numpy as np


### EDA

1. LOAD DATA

In [4]:
# load data
data = pd.read_csv('data_2024_reduced_columns.csv')
data['B2'].unique()

array(['Doing okay', 'Living comfortably', 'Just getting by',
       'Finding it difficult to get by'], dtype=object)

2. CATEGORICAL VARIABLE TRANSFORMATON TO ORDINAL

In [132]:
# map columms from categorial to ordianal
B2_mapping = {
    "Finding it difficult to get by": 1,
    "Just getting by": 2,
    "Doing okay": 3,
    "Living comfortably": 4
}

A6_mapping = {
    "Not confident": 1,
    "Somewhat confident": 2,
    "Don’t know": 3,
    "Very confident": 4
}

C4A_mapping = {
    "Never carried an unpaid balance (always pay in full)": 0,
    "Once": 1,
    "Some of the time": 2,
    "Most or all of the time": 3
}
ppemploy_mapping = {
    "Working part-time": 2,
    "Not working": 3,
    "Working full-time": 1
}

BNPL1_mapping = {
  "No": 0,
  "Yes": 1
}

In [139]:
data['ppemploy'].unique()

array(['Working part-time', 'Not working', 'Working full-time'],
      dtype=object)

In [140]:
# convert columns to ordinal
data["B2_ord"] = data["B2"].map(B2_mapping)
data["A6_ord"] = data["A6"].map(A6_mapping)
data["C4A_ord"] = data["C4A"].map(C4A_mapping)
data["ppemploy_ord"] = data["ppemploy"].map(ppemploy_mapping)
data["BNPL1_ord"] = data["BNPL1"].map(BNPL1_mapping)
data["BNPL1_ord"] = pd.to_numeric(data["BNPL1_ord"], errors="coerce")

In [141]:
variables = ['B2_ord','A6_ord','C4A_ord','ppemploy_ord']
print(data[variables + ["BNPL1_ord"]].dtypes)

B2_ord            int64
A6_ord            int64
C4A_ord         float64
ppemploy_ord      int64
BNPL1_ord         int64
dtype: object


In [142]:
# spearman relationship
spearman_results = []

variables = ['B2_ord','A6_ord','C4A_ord','ppemploy_ord']

for var in variables:
  # spearman correlation
  corr = spearmanr(data[var], data["BNPL1_ord"], nan_policy="omit")[0]
  p = spearmanr(data[var], data["BNPL1_ord"], nan_policy="omit")[1]
  spearman_results.append({
    "variable": var,
    "corr": corr,
    "p_value": p
    })

# convert to DataFrame for display
results_df = pd.DataFrame(spearman_results)
results_df

,variable,corr,p_value
0,B2_ord,-0.182924,5.549600e-93
1,A6_ord,-0.211765,1.137539e-124
2,C4A_ord,0.273239,1.313590e-176
3,ppemploy_ord,-0.074669,1.130028e-16


3. ONE-HOT ENCODE TRANSFORMATION

In [ ]:
B2_dummies = pd.get_dummies(data["B2"], prefix="B2")

# Clean column names
B2_dummies.columns = (
    B2_dummies.columns
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("to_get_by", "to_get_by")  # optional tweak
)

data = pd.concat([data, B2_dummies], axis=1)

,B2,B2_Doing okay
0,Doing okay,True
1,Living comfortably,False
6,Just getting by,False
12,Finding it difficult to get by,False


4. CREATE NEW COLUMNS 

In [145]:
# create column employment_flag
data["employment_flag"] = np.where(
    data['ppemploy'].isna(),  # all missing → keep NaN
    np.nan,
    data['ppemploy'].isin(["Working full-time", "Working part-time"]).astype(int)
)

In [148]:
# combine I0_abcdef to column financial_concern_flag
cols = ["I0_a", "I0_b", "I0_c", "I0_d", "I0_e", "I0_f"]

data["income_flag"] = np.where(
    data[cols].isna().all(axis=1),  # all missing → keep NaN
    np.nan,
    data[cols].isin(["Yes"]).any(axis=1).astype(int)
)

In [42]:
# combine x12_abcdefg to column financial_concern_flag
cols = ["X12_a", "X12_b", "X12_c", "X12_d", "X12_e", "X12_f", "X12_g"]

data["financial_concern_flag"] = np.where(
    data[cols].isna().all(axis=1),  # all missing → keep NaN
    np.nan,
    data[cols].isin(["Major concern", "Minor concern"]).any(axis=1).astype(int)
)

In [47]:
# create new column financial_concern_count to add number of conerns in X12 a to g
data["financial_concern_count"] = np.where(
    data[cols].isna().all(axis=1),
    np.nan,
    data[cols].isin(["Minor concern", "Major concern"]).sum(axis=1)
)

In [53]:
# create new column financial_concern_score to add up number of conerns in X12 a to g with major concer=2, minor concern = 1
mapping = {
    "Not a concern": 0,
    "Minor concern": 1,
    "Major concern": 2
}

data["financial_concern_score"] = np.where(
    data[cols].isna().all(axis=1),
    np.nan,
    data[cols].replace(mapping).sum(axis=1)
)

/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/1170674784.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[cols].replace(mapping).sum(axis=1)


In [58]:
# create financial concern flag for each question X12 abcdefg
mapping = {
    "Not a concern": 0,
    "Minor concern": 1,
    "Major concern": 1
}

cols = ["X12_a", "X12_b", "X12_c", "X12_d", "X12_e", "X12_f", "X12_g"]

for col in cols:
    data[f"financial_concern_flag_{col}"] = np.where(
    data[col].isna(),
    np.nan,
    data[col].replace(mapping)
    )


/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/2038383620.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col].replace(mapping)
/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/2038383620.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col].replace(mapping)
/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/2038383620.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitl

In [15]:
# create financial concern score for each question X12 abcdefg
mapping = {
    "Not a concern": 0,
    "Minor concern": 1,
    "Major concern": 2
}

cols = ["X12_a", "X12_b", "X12_c", "X12_d", "X12_e", "X12_f", "X12_g"]

for col in cols:
    data[f"financial_concern_{col}"] = data[col].replace(mapping)

/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/1267505192.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[f"financial_concern_{col}"] = data[col].replace(mapping)
/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/1267505192.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[f"financial_concern_{col}"] = data[col].replace(mapping)
/var/folders/cr/df6gvtx90rz5bg2m0clb1gym0000gn/T/ipykernel_69592/1267505192.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will

In [66]:
# create renting flag

cols = ['R1_a','R1_b','R1_c','R1_d','R1_e','R1_f','R1_g']

for col in cols:
    data["renting_flag"] = np.where(
        data[cols].isna().all(axis=1),  # all missing → 0
        0,
        (data[cols]=='Yes').any(axis=1).astype(int)
    )

In [82]:
# create credit_card_application_challenge_flag

cols = ['A1_a','A1_b','A1_c']

for col in cols:
    data["credit_card_application_challenge_flag"] = np.where(
        data[cols].isna().all(axis=1),  # all missing → na
        np.nan,
        (data[cols]=='Yes').any(axis=1).astype(int)
    )

In [91]:
# create loan_creditdard_application_failure_flag

cols = ['A8_a','A8_b','A8_c','A8_d','A8_e','A8_f']

for col in cols:
    data["loan_application_failure_flag"] = np.where(
        data[cols].isna().all(axis=1),  # all missing → na
        np.nan,
        (data[cols]=='Yes').any(axis=1).astype(int)
    )

5. CHI-SQUARE TEST

In [149]:
# define cramers_v function
def compute_cramers_v(chi2, table):
    n = table.values.sum()
    r, c = table.shape
    k = min(r, c)
    return np.sqrt(chi2 / (n * (k - 1)))

# variables to test against BNPL1
predictors = ["B2",
              "financial_concern_flag", 'financial_concern_count', 'financial_concern_score',
              'financial_concern_X12_a', 'financial_concern_X12_b', 'financial_concern_X12_c', 'financial_concern_X12_d', 'financial_concern_X12_e', 'financial_concern_X12_f', 'financial_concern_X12_g',
              'financial_concern_flag_X12_a', 'financial_concern_flag_X12_b', 'financial_concern_flag_X12_c', 'financial_concern_flag_X12_d', 'financial_concern_flag_X12_e', 'financial_concern_flag_X12_f', 'financial_concern_flag_X12_g',
              'ppgender',
              'ppethm',
              'ppmarit5',
              'D22_i',
              'renting_flag',
              'credit_card_application_challenge_flag',
              'A8_a','loan_application_failure_flag',
              'EF5C','SL6',
              'SL4A',
              'employment_flag','income_flag']

results = []

for var in predictors:
    # remove na
    subset = data[[var, "BNPL1"]].dropna()

    # contingency table
    table = pd.crosstab(subset[var], subset["BNPL1"])

    # chi-square test
    chi2, p, dof, expected = chi2_contingency(table)

    # cramér's v
    v = compute_cramers_v(chi2, table)

    results.append({
        "variable": var,
        "chi2": chi2,
        "p_value": p,
        "cramers_v": v
    })

# convert to DataFrame for display
results_df = pd.DataFrame(results)
(results_df)


,variable,chi2,p_value,cramers_v
0,B2,424.538073,1.070648e-91,0.185821
1,financial_concern_flag,27.115867,1.916187e-07,0.066536
2,financial_concern_count,175.799385,1.501158e-34,0.169416
3,financial_concern_score,219.365463,5.930431e-39,0.189248
4,financial_concern_X12_a,56.043810,6.764587e-13,0.095656
5,financial_concern_X12_b,68.986489,1.046584e-15,0.106128
6,financial_concern_X12_c,107.524868,4.479934e-24,0.132496
7,financial_concern_X12_d,66.897785,2.973929e-15,0.104509
8,financial_concern_X12_e,228.753785,2.122007e-50,0.193255
9,financial_concern_X12_f,67.356105,2.364872e-15,0.104866


6. Crosstab results for important predictors

In [88]:
# credit card application challenge flag
pd.crosstab(subset['credit_card_application_challenge_flag'], subset["BNPL1"], normalize='index')

BNPL1,No,Yes
credit_card_application_challenge_flag,,
0.0,0.857980,0.142020
1.0,0.603336,0.396664
